# Module 13.3 — Complete NLP Classification Pipeline 🧠

The goal of this module is to understand how raw text becomes a trained NLP model that can make predictions on unseen text.

We'll use sentiment analysis as our running example and follow your exact syllabus sequence:

#### 1

### Dataset

Collect labeled text

#### 2

### Cleaning

Prepare the text

#### 3

### Tokenization

Split text into tokens

#### 4

### Representation

Convert text into numerical features

#### 5

### Model

Choose a classifier

#### 6

### Training

Learn from labeled examples

#### 7

### Evaluation

Measure generalization

## 1. Dataset — collecting labeled text



A dataset contains input text and, for supervised learning, the correct label associated with each example.

| Review                                    | Sentiment |
| ----------------------------------------- | --------- |
| "This product is amazing!"                | Positive  |
| "The quality is terrible."                | Negative  |
| "The package arrived today."              | Neutral   |
| "Excellent performance and battery life." | Positive  |

Here:

- Input \\(X\\): the review text.
- Target \\(y\\): its sentiment label.

The model learns a mapping:

\\[ f(X)\rightarrow \hat{y} \\]

where \\(\hat{y}\\) is the predicted sentiment.

A real dataset needs enough examples to represent different writing styles, classes and edge cases. Labels should follow a consistent annotation policy.

### Split the data

Training set — 70–80%

Used to learn model parameters.

Validation set — 10–15%

Used for model selection and tuning.

Test set — 10–15%

Used for final, unbiased evaluation.

These are illustrative proportions, not mandatory rules. A common starting point is 80/10/10.

Important: Split before fitting the vocabulary or TF-IDF vectorizer. Otherwise, information from the test set can leak into training.

## 2. Cleaning — preparing text

Raw text often contains unwanted formatting, inconsistent whitespace, HTML, URLs or spelling variations.

Example:

```
Original:  "   THIS phone is AMAZING!!! <br> "
Cleaned:   "this phone is amazing!!!"
```

Possible cleaning operations:

- Normalize whitespace.
- Remove unwanted HTML markup.
- Handle URLs and special characters when appropriate.
- Normalize case when capitalization is not useful.
- Preserve meaningful words such as `not`, `never` and `no`.

Important: Cleaning is task-dependent. Removing punctuation, emojis or capitalization can discard sentiment information. For Transformer models, avoid aggressive cleaning unless experiments show that it helps.

## 3. Tokenization — splitting text

Tokenization converts text into smaller units called tokens.

Example:

```
Text: "I love natural language processing"

Tokens:
["I", "love", "natural", "language", "processing"]
```

Classical NLP commonly uses word tokenization. Transformer models usually use subword tokenization, where a word may become multiple tokens.

For example, the same word may be represented as one token in one vocabulary but multiple subword tokens in another.

Why it matters: Tokenization determines which units the model receives and affects vocabulary size, sequence length and how rare words are handled.

## 4. Representation — converting text into numbers

Machines need numerical input, so tokens must be converted into numerical features.



Three important approaches:

| Representation | How it works                                       | Typical use                 |
| -------------- | -------------------------------------------------- | --------------------------- |
| Bag of Words   | Counts vocabulary terms                            | Simple text classification  |
| TF-IDF         | Weights terms by their importance across documents | Classical NLP baselines     |
| Embeddings     | Represents tokens or text as dense vectors         | Neural NLP and Transformers |

Example vocabulary:

```
["bad", "excellent", "good"]
```

The sentence `"good excellent good"` could have this Bag of Words representation:

\\[ X=[0,1,2] \\]

The entries correspond to the counts of `bad`, `excellent` and `good`.

TF-IDF would instead assign weights based on term frequency and inverse document frequency.

Connection to your completed modules: You already covered BoW, TF-IDF and pretrained embeddings. The key here is understanding where each representation fits into the pipeline.

## 5. Model — choosing a classifier

Once the text is represented numerically, we choose a model to predict its sentiment.



For your syllabus, the primary baseline models are:

- Logistic Regression: a strong starting point for TF-IDF features.
- Naive Bayes: a fast probabilistic baseline.
- Linear SVM: often effective for sparse, high-dimensional text features.

For three mutually exclusive sentiment classes, the classifier produces a prediction among positive, negative and neutral.

The important decision is not to select the most complicated model automatically. Start with a baseline, evaluate it, then justify any added complexity.

## 6. Training — learning from examples

Training adjusts the model parameters using labeled examples.

For a multiclass classifier, the process is:

1. Provide a numerical representation of a review.
2. Produce predicted class scores.
3. Compare the predictions with the true labels using a loss function.
4. Update model parameters to reduce the loss.
5. Repeat over the training data.

Conceptually:

\\[ \theta^\*=\arg\min\_\theta \mathcal{L}(f\_\theta(X),y) \\]

Here, \\(\theta\\) represents the learnable parameters and \\(\mathcal{L}\\) is the training loss.

For a classical TF-IDF + Logistic Regression model, the vectorizer constructs features and the classifier learns its parameters. You do not need to train a neural network or use CUDA for this baseline.

For Transformer fine-tuning later, PyTorch and CUDA become useful when the model and workload benefit from GPU acceleration.

## 7. Evaluation — measuring model performance

A model performing well on its training data does not guarantee that it will perform well on new reviews.

Evaluate the trained model on data it has not seen during training.

### Important metrics

| Metric           | Purpose                                       |
| ---------------- | --------------------------------------------- |
| Accuracy         | Overall fraction of correct predictions       |
| Precision        | How reliable predictions for a class are      |
| Recall           | How many actual examples of a class are found |
| F1-score         | Balance between precision and recall          |
| Confusion matrix | Shows which classes are confused              |

For a three-class sentiment problem, macro F1-score is useful when each class deserves equal importance, particularly when the dataset is imbalanced.

A confusion matrix can reveal, for example, that the model frequently predicts neutral reviews as positive.

---

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

texts = [
    "Amazing product and excellent quality",
    "Terrible product and very poor quality",
    "The package arrived today",
    "I really love this product",
    "I hate this product",
    "The delivery happened yesterday",
    "Excellent performance and great design",
    "Awful performance and disappointing design",
    "The product has two buttons",
]

labels = [
    "positive", "negative", "neutral",
    "positive", "negative", "neutral",
    "positive", "negative", "neutral",
]

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels,
)

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    zero_division=0,
))

print(model.predict([
    "The product is excellent",
    "This is a terrible purchase",
]))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
     neutral       1.00      1.00      1.00         1
    positive       0.50      1.00      0.67         1

    accuracy                           0.67         3
   macro avg       0.50      0.67      0.56         3
weighted avg       0.50      0.67      0.56         3

['neutral' 'negative']
